# PhoBERT ESG Actionability Classification
This notebook is optimized for Colab / Kaggle. Make sure to upload the `data/labels` directory before running, or mount your Google Drive.

In [ ]:
# Install dependencies for Colab/Kaggle
!pip install transformers pandas pyarrow scikit-learn accelerate

In [ ]:
# Configuration
CONFIG = {
    "model": {
        "name": "vinai/phobert-large",
        "max_length": 256
    },
    "training": {
        "epochs": 5,
        "batch_size": 16,
        "learning_rate": 2.0e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "use_class_weights": True,
        "weight_method": "balanced" # options: balanced for sklearn default
    },
    "paths": {
        # Upload your data/labels folder to Kaggle/Colab, or mount Google Drive
        "train_data": "data/labels/action_hybrid_train_split.parquet",
        "val_data": "data/labels/action_hybrid_val_split.parquet",
        "gold_test": "data/labels/action_gold.parquet",
        "output_dir": "outputs/models/action_phobert_large",
        "inference_input": "data/corpus/esg_sentences.parquet",
        "inference_weak": "data/labels/action_weak.parquet",
        "inference_output": "data/corpus/esg_sentences_with_actionability.parquet"
    }
}


In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from tqdm import tqdm
import argparse

MODEL_NAME = CONFIG["model"]["name"]
OUTPUT_DIR = Path(CONFIG["paths"]["output_dir"])
LABELS = ["Implemented", "Planning", "Indeterminate"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for i, label in enumerate(LABELS)}

CONTEXT_BLOCK_TYPES = {"paragraph", "bullet_like", "kpi_like"}

class ActionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 256, use_context: bool = True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_context = use_context
        
        self.texts = []
        for _, row in df.iterrows():
            if use_context and row.get("block_type", "") in CONTEXT_BLOCK_TYPES:
                parts = []
                if pd.notna(row.get("ctx_prev")) and row["ctx_prev"]:
                    parts.append(str(row["ctx_prev"]))
                parts.append(str(row["sentence"]))
                if pd.notna(row.get("ctx_next")) and row["ctx_next"]:
                    parts.append(str(row["ctx_next"]))
                text = " ".join(parts)
            else:
                text = str(row["sentence"])
            self.texts.append(text)
        
        label_col = "final_action" if "final_action" in df.columns else "gold_action"
        if label_col in df.columns:
            self.labels = [LABEL2ID[label] for label in df[label_col]]
        else:
            self.labels = None
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        if self.class_weights is not None:
            weight = torch.tensor(self.class_weights, device=logits.device, dtype=logits.dtype)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        
        loss = loss_fct(logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    per_class_f1 = f1_score(labels, predictions, average=None)
    
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "f1_implemented": per_class_f1[0],
        "f1_planning": per_class_f1[1],
        "f1_indeterminate": per_class_f1[2],
    }


In [ ]:
def train(config):
    epochs = config["training"]["epochs"]
    batch_size = config["training"]["batch_size"]
    lr = config["training"]["learning_rate"]
    max_len = config["model"]["max_length"]
    use_weights = config["training"]["use_class_weights"]
    
    print("Loading training data...")
    train_df = pd.read_parquet(config["paths"]["train_data"])
    val_df = pd.read_parquet(config["paths"]["val_data"])
    
    print(f"Train: {len(train_df)}, Val: {len(val_df)}")
    print(f"Train distribution:\n{train_df['final_action'].value_counts()}")
    
    class_weights = None
    if use_weights:
        labels_array = train_df["final_action"].map(LABEL2ID).values
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.array([0, 1, 2]),
            y=labels_array
        )
        print(f"\nClass weights: {dict(zip(LABELS, class_weights))}")
    
    print(f"\nLoading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    
    train_dataset = ActionDataset(train_df, tokenizer, max_len)
    val_dataset = ActionDataset(val_df, tokenizer, max_len)
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        weight_decay=config["training"]["weight_decay"],
        warmup_ratio=config["training"]["warmup_ratio"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    
    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    
    print("\nStarting training...")
    trainer.train()
    
    print("\n" + "="*60)
    print("VALIDATION RESULTS")
    print("="*60)
    val_results = trainer.evaluate()
    for k, v in val_results.items():
        print(f"  {k}: {v:.4f}")
        
    gold_path = Path(config["paths"]["gold_test"])
    if gold_path.exists():
        print("\n" + "="*60)
        print("GOLD SET EVALUATION")
        print("="*60)
        gold_df = pd.read_parquet(gold_path)
        if "gold_action" in gold_df.columns:
            gold_dataset = ActionDataset(gold_df, tokenizer, max_len)
            gold_predictions = trainer.predict(gold_dataset)
            gold_preds = np.argmax(gold_predictions.predictions, axis=-1)
            gold_labels = [LABEL2ID[l] for l in gold_df["gold_action"]]
            
            print(f"  Gold Accuracy: {accuracy_score(gold_labels, gold_preds):.4f}")
            print(f"  Gold Macro-F1: {f1_score(gold_labels, gold_preds, average='macro'):.4f}")
            print("\nClassification Report (Gold):")
            print(classification_report(gold_labels, gold_preds, target_names=LABELS))
            
    final_path = OUTPUT_DIR / "final"
    trainer.save_model(str(final_path))
    tokenizer.save_pretrained(str(final_path))
    print(f"\nModel saved: {final_path}")
    
    return tokenizer, model


In [ ]:
def batch_inference(texts, tokenizer, model, batch_size=32, max_len=256, device="cuda"):
    all_preds = []
    all_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
        batch_texts = texts[i:i + batch_size]

        encoding = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )
        encoding = {k: v.to(device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return all_preds, all_probs


def run_inference(config, tokenizer=None, model=None):
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Device: {device}")
    
    if tokenizer is None or model is None:
        model_path = Path(config["paths"]["output_dir"]) / "final"
        print(f"Loading model from {model_path}...")
        tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
    
    model.to(device)
    model.eval()

    print("Loading sentences for inference...")
    df = pd.read_parquet(config["paths"]["inference_input"])
    print(f"Total sentences: {len(df)}")
    
    # Use ActionDataset logic for consistency
    dataset_for_inference = ActionDataset(df, tokenizer, config["model"]["max_length"], use_context=True)
    texts = dataset_for_inference.texts
    
    batch_size = config["training"]["batch_size"] * 2
    preds, probs = batch_inference(texts, tokenizer, model, batch_size, config["model"]["max_length"], device)
    
    df["action_pred_raw"] = [ID2LABEL[p] for p in preds]
    df["action_prob_raw"] = [max(p) for p in probs]
    df["action_probs"] = probs
    
    df["action_pred"] = df["action_pred_raw"]
    df["action_prob"] = df["action_prob_raw"]
    df["override_reason"] = ""
    
    weak_path = Path(config["paths"]["inference_weak"])
    if weak_path.exists():
        weak_df = pd.read_parquet(weak_path)
        print(f"Loaded weak labels for override: {len(weak_df)}")
        weak_subset = weak_df[["sent_id", "weak_action", "weak_conf"]].copy()
        df = df.merge(weak_subset, on="sent_id", how="left")

        # Example override: if weak label has very high confidence and model is uncertain
        mask_override = (
            (df["weak_conf"] >= 0.8) &
            (df["action_prob_raw"] < 0.6) &
            (df["override_reason"] == "")
        )
        df.loc[mask_override, "action_pred"] = df.loc[mask_override, "weak_action"]
        df.loc[mask_override, "action_prob"] = df.loc[mask_override, "weak_conf"]
        df.loc[mask_override, "override_reason"] = "weak_override"
        
        df = df.drop(columns=["weak_action", "weak_conf"])
        
    df["action_probs"] = df["action_probs"].apply(
        lambda p: {ID2LABEL[i]: round(v, 4) for i, v in enumerate(p)}
    )
    df = df.drop(columns=["action_pred_raw", "action_prob_raw"])
    
    output_path = Path(config["paths"]["inference_output"])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    
    print(f"\n=== INFERENCE COMPLETE ===")
    print(f"Output saved to: {output_path}")
    print(f"\nActionability Distribution:")
    print(df["action_pred"].value_counts())
    
    return df


In [ ]:
# 1. Train the model
trained_tokenizer, trained_model = train(CONFIG)

In [ ]:
# 2. Run inference on the entire corpus
inferred_df = run_inference(CONFIG, trained_tokenizer, trained_model)
inferred_df.head()